In [2]:


import weaviate
from weaviate.classes.config import Configure
from weaviate.classes.query import MetadataQuery
from weaviate.classes.tenants import Tenant


In [10]:
# Test connection to local Weaviate instance
client = weaviate.connect_to_local()
print(client.is_ready())
client.close()

True


In [11]:
# Create a collection

client = weaviate.connect_to_local()

client.collections.delete("DemoCollection")  # delete if it already exists

demo_collection = client.collections.create(
    name="DemoCollection",  # Name of the collection
    multi_tenancy_config=Configure.multi_tenancy(
        enabled=True,
        auto_tenant_creation=True
    ),
    vector_config=[
        Configure.Vectors.text2vec_openai(
            name="title_vector",  # Name of the vector property
            source_properties=["title"],  # Which properties of the source object to use for vectorization
            model="text-embedding-3-large",
            dimensions=None  # Let Weaviate infer the dimensions from the model
        )
    ]
)

demo_collection.tenants.create(
    tenants=[
        Tenant(name="tenantA"),
        Tenant(name="tenantB"),
    ]
)

#print(demo_collection.config.get(simple=True))

client.close()

In [14]:
# import data into the collection in batches
source_objects = [
    {"title": "The Shawshank Redemption",
     "description": "A wrongfully imprisoned man forms an inspiring friendship while finding hope and redemption in the darkest of places."},
    {"title": "The Godfather",
     "description": "A powerful mafia family struggles to balance loyalty, power, and betrayal in this iconic crime saga."},
    {"title": "The Dark Knight",
     "description": "Batman faces his greatest challenge as he battles the chaos unleashed by the Joker in Gotham City."},
    {"title": "Jingle All the Way",
     "description": "A desperate father goes to hilarious lengths to secure the season's hottest toy for his son on Christmas Eve."},
    {"title": "A Christmas Carol",
     "description": "A miserly old man is transformed after being visited by three ghosts on Christmas Eve in this timeless tale of redemption."}
]
client = weaviate.connect_to_local()
collection = client.collections.get("DemoCollection").with_tenant("tenantA")

with collection.batch.fixed_size(batch_size=200) as batch:
    for src_obj in source_objects:
        # The model provider integration will automatically vectorize the object
        batch.add_object(
            properties={
                "title": src_obj["title"],
                "description": src_obj["description"],
            },
            # vector=vector  # Optionally provide a pre-obtained vector
        )
        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.")
            break

failed_objects = collection.batch.failed_objects
if failed_objects:
    print(f"Number of failed imports: {len(failed_objects)}")
    print(f"First failed object: {failed_objects[0]}")

client.close()

In [15]:
# Query the collection
client = weaviate.connect_to_local()
collection = client.collections.get("DemoCollection").with_tenant("tenantA")

response = collection.query.near_text(
    query="A super hero film",  # The model provider integration will automatically vectorize the query
    limit=2,
    return_metadata=MetadataQuery(score=True, distance=True)
)

for obj in response.objects:
    print(obj.properties["title"])

client.close()

The Dark Knight
The Shawshank Redemption
